# 06 — Implantação e encerramento da segmentação de RH
**CRISP-DM — Fase 6 | IBM HR Analytics**

Objetivo: verificar a aprovação registrada no notebook 05, documentar o contrato de entrada, testar a reprodução da partição, planejar o monitoramento e registrar o encerramento.

**Limitação:** o notebook 03 não persiste o pré-processador. O pacote, quando autorizado, aceita somente a matriz numérica já preparada — não dados brutos de novos colaboradores. Nenhuma decisão individual automatizada é proposta.

## 0. Arquivos e importações

In [1]:
from pathlib import Path
import hashlib
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score

DADOS = Path('../data/processed')
MODELOS = Path('../models')
TABELAS = Path('../reports/tables/rh')
MODELOS.mkdir(parents=True, exist_ok=True)
TABELAS.mkdir(parents=True, exist_ok=True)

caminhos = {
    'matriz': DADOS / 'base_rh_preprocessed.csv',
    'segmentos': DADOS / 'segmentos_rh.csv',
    'selecao': MODELOS / 'selecionado_rh.json',
    'avaliacao': MODELOS / 'avaliacao_rh.json',
    'auditoria': MODELOS / 'auditoria_rh_05.json',
}
for arquivo in caminhos.values():
    if not arquivo.is_file():
        raise FileNotFoundError(f'Arquivo ausente: {arquivo}. Execute os notebooks anteriores.')

X = pd.read_csv(caminhos['matriz'], sep=';')
segmentos = pd.read_csv(caminhos['segmentos'], sep=';')
selecao = json.loads(caminhos['selecao'].read_text(encoding='utf-8'))
avaliacao = json.loads(caminhos['avaliacao'].read_text(encoding='utf-8'))
auditoria = json.loads(caminhos['auditoria'].read_text(encoding='utf-8'))
print('Matriz:', X.shape, '| Avaliação:', avaliacao['decisao'])

FileNotFoundError: Arquivo ausente: ../models/avaliacao_rh.json. Execute os notebooks anteriores.

## 1. Planejar implantação
Conferir que os dados e o modelo são aqueles avaliados no notebook 05. O parecer `aprovado` é obrigatório para gerar um pacote.

In [ ]:
if len(X) != len(segmentos) or 'segmento' not in segmentos:
    raise ValueError('Matriz e segmentos incompatíveis')
if X.columns.tolist() != selecao['atributos'] or X.shape != (selecao['n_amostras'], selecao['n_atributos']):
    raise ValueError('Matriz diferente da selecionada no notebook 04')
if X.isna().any().any() or not np.isfinite(X.to_numpy(dtype=float)).all():
    raise ValueError('Matriz contém valores ausentes ou não finitos')
if avaliacao['modelo_arquivo'] != selecao['arquivo']:
    raise ValueError('O modelo avaliado difere do selecionado')

for nome in ['matriz', 'segmentos', 'selecao']:
    arquivo = caminhos[nome]
    hash_atual = hashlib.sha256(arquivo.read_bytes()).hexdigest()
    if avaliacao['entradas_sha256'].get(arquivo.name) != hash_atual:
        raise ValueError(f'Arquivo alterado após a avaliação: {arquivo.name}')

if auditoria['n_registros'] != len(X) or auditoria['n_atributos'] != X.shape[1]:
    raise ValueError('Auditoria do notebook 05 incompatível com a matriz')

APROVADO = avaliacao['decisao'] == 'aprovado'
print('Autorização para empacotamento:', APROVADO)

In [ ]:
contrato = {
    'versao': '0.1.0',
    'tipo_entrada': 'matriz_preprocessada',
    'colunas_em_ordem': X.columns.tolist(),
    'n_atributos': X.shape[1],
    'faixa_esperada': [0.0, 1.0],
    'transformador_bruto_persistido': False,
    'modelo': selecao['modelo'],
}
(MODELOS / 'contrato_rh.json').write_text(
    json.dumps(contrato, indent=2, ensure_ascii=False), encoding='utf-8'
)

arquivo_modelo = MODELOS / selecao['arquivo']
if not arquivo_modelo.is_file():
    raise FileNotFoundError(arquivo_modelo)
modelo = joblib.load(arquivo_modelo)  # Usar somente artefatos próprios e confiáveis.

reproducao = {'disponivel': hasattr(modelo, 'predict'), 'ari': None, 'particao_reproduzida': False}
if reproducao['disponivel']:
    previsto = modelo.predict(X.to_numpy(dtype=float))
    reproducao['ari'] = float(adjusted_rand_score(segmentos['segmento'], previsto))
    reproducao['particao_reproduzida'] = bool(np.isclose(reproducao['ari'], 1.0))

print('Teste de reprodução:', reproducao)

## 2. Planejar monitoramento e manutenção
Indicadores propostos para discussão com RH; nenhum limite operacional foi previamente aprovado.

In [ ]:
monitoramento = pd.DataFrame([
    {'indicador': 'Tamanho dos segmentos', 'frequencia': 'Por lote', 'criterio': 'A definir com RH'},
    {'indicador': 'Qualidade dos dados de entrada', 'frequencia': 'Por lote', 'criterio': 'A definir com RH'},
    {'indicador': 'Mudança na distribuição dos atributos', 'frequencia': 'Mensal (proposta)', 'criterio': 'Definir baseline'},
    {'indicador': 'Silhueta dos segmentos', 'frequencia': 'Em reavaliações', 'criterio': 'Comparar com baseline'},
    {'indicador': 'Desligamento por segmento', 'frequencia': 'Trimestral (proposta)', 'criterio': 'Análise descritiva e humana'},
])
monitoramento.to_csv(TABELAS / '06_monitoramento_rh.csv', sep=';', index=False)
display(monitoramento)

## 3. Relatório final e pacote
Registrar o resultado da implantação. Só criar o pacote se a avaliação tiver sido aprovada e o modelo reproduzir a partição.

In [ ]:
relatorio = {
    'fase': 'implantacao',
    'decisao_avaliacao': avaliacao['decisao'],
    'modelo': selecao['modelo'],
    'n_registros': len(X),
    'n_atributos': X.shape[1],
    'silhueta_avaliada': avaliacao['silhueta'],
    'reproducao': reproducao,
    'contrato': 'contrato_rh.json',
    'monitoramento': 'reports/tables/rh/06_monitoramento_rh.csv',
    'limitacoes': [
        'Sem pré-processador persistido para dados brutos',
        'Critérios de aceitação de RH ainda não formalizados',
        'Sem decisões individuais automatizadas',
    ],
}
(MODELOS / 'relatorio_implantacao_rh.json').write_text(
    json.dumps(relatorio, indent=2, ensure_ascii=False), encoding='utf-8'
)

pacote = MODELOS / 'pacote_segmentacao_rh.joblib'
if APROVADO and reproducao['particao_reproduzida']:
    joblib.dump({'modelo': modelo, 'contrato': contrato, 'avaliacao': avaliacao, 'relatorio': relatorio}, pacote)
    print('Pacote gerado:', pacote)
else:
    # Evitar deixar pacote aprovado de uma execução anterior no diretório.
    if pacote.exists():
        pacote.unlink()
    print('Pacote não publicado: avaliação pendente/rejeitada ou reprodução não validada.')

## 4. Revisão do projeto e próximas etapas
Registrar as restrições que impedem uso operacional com dados brutos e as decisões pendentes.

In [ ]:
encerramento = {
    'status': 'avaliacao_aprovada' if APROVADO else 'pronto_para_revisao',
    'pacote_publicado': pacote.is_file() and APROVADO and reproducao['particao_reproduzida'],
    'pendencias': [
        'Aprovar critérios de avaliação com RH',
        'Persistir o pré-processador do notebook 03',
        'Validar classificação ponta a ponta para novos registros',
        'Definir responsáveis e limites de monitoramento',
        'Avaliar estabilidade dos clusters',
    ],
}
(MODELOS / 'encerramento_rh.json').write_text(
    json.dumps(encerramento, indent=2, ensure_ascii=False), encoding='utf-8'
)
print('Status:', encerramento['status'], '| Pacote publicado:', encerramento['pacote_publicado'])